<a href="https://colab.research.google.com/github/aaditjoshi-star/Aadit/blob/main/Exercise3_Self_Reflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 3: Self-Reflection Prompt for Improving Output
**Aadit Joshi** | Agentic AI and Distributed Systems | San José State University | Fall 2026

**Goal:** Have an AI model write a summary, then critique its own summary against explicit requirements and produce an improved version (critique-and-revise).

**Tools used**
- Google Colab (Python 3)
- Gemini API through the `google-genai` Python SDK, free tier (first choice `gemini-3.8-flash`, with other free Flash models as fallbacks if a daily limit is hit)
- Plain Python for automatic requirement checks, and `pandas` for the before/after comparison table

**How to run**
1. Create a free API key at https://aistudio.google.com/apikey
2. In Colab, open **Secrets** (key icon in the left sidebar), add `GEMINI_API_KEY`, and turn on notebook access.
3. Runtime > Run all.

## Flow

```
 Source report --> DRAFT prompt --> "Before" summary
                                          |
                                          v
                   SELF-REFLECTION prompt: check the summary against requirements R1-R5,
                   give PASS/FAIL + evidence + fix for each, then write a revised summary
                                          |
                                          v
                   Automatic checks in code --> all pass: "After" summary
                                            --> any fail: problems go back for one more reflection round
```

## Setup

In [2]:
!pip install -q google-genai

In [3]:
# Setup: Gemini client and shared helper functions
import os, json, re, time, textwrap
from google import genai
from google.genai import types, errors

# API key is stored in Colab Secrets (key icon in the left sidebar) as GEMINI_API_KEY
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY").strip()
except ImportError:
    pass  # not running in Colab: set the GEMINI_API_KEY environment variable instead

client = genai.Client()  # reads GEMINI_API_KEY

# First-choice model plus free-tier fallbacks (each model has its own daily quota)
MODELS = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash",
          "gemini-3.5-flash", "gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]
model_index = 0
MODELS_USED = set()
RETRYABLE = {429, 500, 503}


def generate(prompt, system=None, json_mode=False, max_retries=4):
    """Send a prompt to Gemini and return the reply text.
    Retries on rate limits / server errors and falls back to the next model if needed."""
    global model_index
    config = types.GenerateContentConfig(
        system_instruction=system,
        response_mime_type="application/json" if json_mode else None,
    )
    attempt = 0
    while True:
        model = MODELS[model_index]
        try:
            response = client.models.generate_content(model=model, contents=prompt, config=config)
            if response.text and response.text.strip():
                MODELS_USED.add(model)
                return response.text.strip()
            code, detail = "empty", "empty response"
        except errors.APIError as e:
            code, detail = e.code, str(e)
            if code in (401, 403):
                raise RuntimeError("Google rejected the API key. Put a current key in the GEMINI_API_KEY "
                                   "secret (key icon, left sidebar), then re-run this setup cell.") from None
            if code not in RETRYABLE and code != 404:
                raise
        attempt += 1
        out_of_daily_quota = code == 429 and "perday" in detail.lower().replace(" ", "")
        if code == 404 or out_of_daily_quota or attempt > max_retries:
            if model_index + 1 >= len(MODELS):
                raise RuntimeError(f"All models failed. Last error: {detail[:300]}")
            model_index += 1
            attempt = 0
            print(f"   [{model} unavailable ({code}); switching to {MODELS[model_index]}]")
        else:
            wait = 15 * attempt
            print(f"   [{model} returned {code}; retrying in {wait}s]")
            time.sleep(wait)


def parse_json(text):
    """Parse a JSON reply, tolerating ```json fences or stray text around the object."""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return json.loads(re.search(r"\{.*\}", cleaned, re.DOTALL).group(0))


def wrap(text, width=100):
    """Wrap long lines so outputs are readable on GitHub (bullets keep a hanging indent)."""
    out = []
    for line in str(text).splitlines():
        indent = "  " if line.startswith("- ") else ""
        out.append(line if len(line) <= width else textwrap.fill(line, width, subsequent_indent=indent))
    return "\n".join(out)


def show(title, body=""):
    print("\n" + "=" * 100)
    print(title)
    print("-" * 100)
    if body != "":
        print(wrap(body) if isinstance(body, str) else json.dumps(body, indent=2))

In [4]:
# Quick connectivity check
print(generate("Reply with exactly one word: ready"))
print("Model in use:", MODELS[model_index])

ready
Model in use: gemini-3.8-flash


## Source document

A fictional quarterly report from the NimbusCart support team (the same made-up store as Exercises 1 and 2). It is long, full of numbers and internal jargon (FRT, SLA, CSAT, FCR, AHT), and ends with a decision leadership has to make. That makes it a good test: a useful summary has to pick the right facts, translate the jargon, and lead to the decision.

In [5]:
REPORT = """NimbusCart Customer Support: Q3 2026 Operations Review
Prepared by: Support Operations team | Distribution: CX leadership

Overview
Q3 (July to September 2026) was the busiest quarter in NimbusCart's history for support. Ticket volume rose to 48,200, up 31% from 36,800 in Q2. The increase came mainly from the launch of the SP-200 smart plug in July and a carrier outage in mid-August that delayed about 6,500 orders.

Performance
- Median first response time (FRT) worsened from 2.1 hours to 5.4 hours. The SLA target is 4 hours, and we missed it in 9 of 13 weeks.
- CSAT fell from 91% to 84%, the lowest since tracking began in 2023. Free-text comments point mostly to slow replies, not to the quality of answers.
- First-contact resolution (FCR) held steady at 72%, and AHT dropped slightly from 11.5 to 10.8 minutes. Agents are not slower; there are simply more tickets than agents.
- Technical tickets about the SP-200 made up 22% of all volume, and 61% of those were the same issue: the plug only works on 2.4 GHz Wi-Fi.

What we tried
In August we piloted an AI assistant that answers SP-200 Wi-Fi questions in the help center before a ticket is created. Over 4 weeks it deflected 38% of SP-200 tickets with no drop in CSAT for those customers. The pilot cost $6,000.
We also moved 5 agents from the returns queue to the technical queue. This shortened technical wait times but pushed returns response time above the SLA in September.

Staffing and costs
The team has 42 agents. Overtime cost $58,000 in Q3, compared with $19,000 in Q2. Two agents resigned in September, citing workload. Hiring and training a new agent takes about 7 weeks and costs roughly $9,000 per hire.

Outlook for Q4
The holiday season usually raises volume by another 25 to 35%. If nothing changes, we project a median first response time above 8 hours in December and CSAT below 80%.

Options
A. Hire 8 new agents now: about $72,000 in hiring and training, productive by mid-November.
B. Roll out the AI assistant to the top 5 ticket topics: about $30,000 to set up plus $4,000 per month. Based on the pilot, it should deflect 20 to 30% of total volume.
C. Do both.

Recommendation
The support operations team recommends a version of option C: expand the AI assistant immediately and hire 5 agents instead of 8, for a total of about $75,000 plus $4,000 per month. We need a decision by October 9 so new hires are ready by the end of November, before peak holiday volume.

Appendix note: the SP-200 firmware team says a fix for the 2.4 GHz setup flow is planned for early November, which should further reduce technical tickets."""

print(f"{len(REPORT.split())} words")
print(REPORT)

459 words
NimbusCart Customer Support: Q3 2026 Operations Review
Prepared by: Support Operations team | Distribution: CX leadership

Overview
Q3 (July to September 2026) was the busiest quarter in NimbusCart's history for support. Ticket volume rose to 48,200, up 31% from 36,800 in Q2. The increase came mainly from the launch of the SP-200 smart plug in July and a carrier outage in mid-August that delayed about 6,500 orders.

Performance
- Median first response time (FRT) worsened from 2.1 hours to 5.4 hours. The SLA target is 4 hours, and we missed it in 9 of 13 weeks.
- CSAT fell from 91% to 84%, the lowest since tracking began in 2023. Free-text comments point mostly to slow replies, not to the quality of answers.
- First-contact resolution (FCR) held steady at 72%, and AHT dropped slightly from 11.5 to 10.8 minutes. Agents are not slower; there are simply more tickets than agents.
- Technical tickets about the SP-200 made up 22% of all volume, and 61% of those were the same issue: 

## Requirements for the summary

These are the criteria the model uses to critique its own work (R1 to R5). The same requirements are also checked automatically in code, so the improvement is measured, not just claimed by the model.

In [6]:
REQUIREMENTS = """R1 Accuracy: Every number, date and claim must come from the report. Write numbers and dates exactly as the report does (for example "$75,000", not "$75K"; "October 9", not "Oct 9"). No invented facts, and no conclusions the report does not state.
R2 Audience and clarity: The reader is Maya Chen, VP of Customer Experience. She is not a support-operations specialist and reads this on her phone between meetings. Use plain English. Do not use the acronyms FRT, SLA, CSAT, FCR or AHT; say what they measure instead (for example "customer satisfaction" instead of "CSAT"). Leave out details she does not need to make the decision.
R3 Length: 120 words or fewer in total.
R4 Format: Plain text only (no headings, no bold, no numbered lists). Use exactly this structure:
  - Line 1 starts with "Bottom line:" followed by one sentence.
  - Then exactly 3 bullet lines. Each starts with "- " and contains at least one number from the report.
  - The last line starts with "Decision needed:" and states the recommended option, its cost and the deadline.
R5 Focus: Cover (a) what went wrong in Q3 and why, (b) what happens in Q4 if nothing changes, and (c) the team's recommendation with its cost and decision deadline."""

BANNED_ACRONYMS = ["FRT", "SLA", "CSAT", "FCR", "AHT"]
NUMBER_PATTERN = r"\d+(?:[.,]\d+)*"


def numbers_in(text):
    return {n.replace(",", "") for n in re.findall(NUMBER_PATTERN, text)}


REPORT_NUMBERS = numbers_in(REPORT)


def check_summary(summary):
    """Automatic checks that mirror R1 to R5. Returns {check: (passed, detail)}."""
    lines = [l.strip() for l in summary.strip().splitlines() if l.strip()]
    bullets = [l for l in lines if l.startswith("- ")]
    words = len(summary.split())
    untraceable = sorted(numbers_in(summary) - REPORT_NUMBERS)
    acronyms = [a for a in BANNED_ACRONYMS if re.search(rf"\b{a}\b", summary)]
    return {
        "R1 every number appears in the report": (not untraceable, f"not in report: {untraceable}" if untraceable else ""),
        "R2 no jargon acronyms": (not acronyms, f"uses {acronyms}" if acronyms else ""),
        "R3 120 words or fewer": (words <= 120, f"{words} words"),
        "R4 starts with 'Bottom line:'": (bool(lines) and lines[0].startswith("Bottom line:"), ""),
        "R4 exactly 3 bullets, each with a number": (
            len(bullets) == 3 and all(re.search(r"\d", b) for b in bullets), f"{len(bullets)} bullets"),
        "R4 ends with 'Decision needed:'": (bool(lines) and lines[-1].startswith("Decision needed:"), ""),
        "R4 plain text (no headings or bold)": (
            "**" not in summary and not any(l.startswith("#") for l in lines), ""),
        "R5 gives the Q4 outlook": ("8 hours" in summary or "80%" in summary, ""),
        "R5 gives the cost ($75,000) and deadline (October 9)": (
            "75,000" in summary and "October 9" in summary, ""),
    }


def failed_checks(summary):
    return [f"{name}" + (f" ({detail})" if detail else "")
            for name, (ok, detail) in check_summary(summary).items() if not ok]


print(REQUIREMENTS)
print(f"\n{len(check_summary('x'))} automatic checks defined.")

R1 Accuracy: Every number, date and claim must come from the report. Write numbers and dates exactly as the report does (for example "$75,000", not "$75K"; "October 9", not "Oct 9"). No invented facts, and no conclusions the report does not state.
R2 Audience and clarity: The reader is Maya Chen, VP of Customer Experience. She is not a support-operations specialist and reads this on her phone between meetings. Use plain English. Do not use the acronyms FRT, SLA, CSAT, FCR or AHT; say what they measure instead (for example "customer satisfaction" instead of "CSAT"). Leave out details she does not need to make the decision.
R3 Length: 120 words or fewer in total.
R4 Format: Plain text only (no headings, no bold, no numbered lists). Use exactly this structure:
  - Line 1 starts with "Bottom line:" followed by one sentence.
  - Then exactly 3 bullet lines. Each starts with "- " and contains at least one number from the report.
  - The last line starts with "Decision needed:" and states the

## Step 1: The "before" summary

A first-draft prompt, the kind most people write: it names the reader but gives no criteria.

In [ ]:
DRAFT_PROMPT = """Summarize this report for Maya Chen, our VP of Customer Experience.

<report>
{report}
</report>"""

before = generate(DRAFT_PROMPT.replace("{report}", REPORT))
show(f"BEFORE: first-draft summary ({len(before.split())} words)", before)

print("\nAutomatic checks on the BEFORE summary:")
for name, (ok, detail) in check_summary(before).items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f"   [{detail}]" if detail else ""))

   [gemini-3.8-flash unavailable (429); switching to gemini-3.7-flash]
   [gemini-3.7-flash returned 503; retrying in 15s]
   [gemini-3.7-flash returned 503; retrying in 30s]


## Step 2: Self-reflection (critique and revise)

The reflection prompt gives the model the report, the requirements, and its own draft. It must:
1. judge the draft against **each** requirement (PASS/FAIL), quoting the evidence and stating the fix,
2. write a revised summary that fixes every FAIL,
3. re-check the revision itself before answering.

The revision is then checked in code. If any automatic check fails, the specific problems are sent back for another reflection round (max 3 rounds).

In [ ]:
REFLECT_SYSTEM = ("You are a meticulous editor who reviews executive summaries against a written checklist. "
                  "You are strict: a requirement passes only if it is fully met. You output valid JSON only.")

REFLECT_PROMPT = """Review the draft summary below against each requirement, then rewrite it.

Steps:
1. For EACH requirement R1 to R5, give a verdict (PASS or FAIL), quote the exact words in the draft that show the problem ("none" if it passes), and state exactly how to fix it.
2. Write a revised summary that fixes every FAIL and keeps what already works. It must meet all five requirements.
3. Before answering, re-check your revised summary against R1 to R5 and correct it if needed. Count the words.

<report>
{report}
</report>

<requirements>
{requirements}
</requirements>

<draft_summary>
{draft}
</draft_summary>
{checker_feedback}
Return JSON with exactly these keys:
{
  "critique": [
    {"requirement": "R1 Accuracy", "verdict": "PASS or FAIL", "evidence": "quoted words or none", "fix": "what to change, or none"}
  ],
  "revised_summary": "the full revised summary as plain text, with \\n between lines"
}
The critique list must have exactly 5 items, one per requirement, in order R1 to R5."""


def reflect(draft, checker_problems=None):
    """One reflection round: critique the draft against R1-R5 and return (critique, revised summary)."""
    feedback = ""
    if checker_problems:
        feedback = ("\n<automatic_checker_results>\nAn automatic checker found these problems in this draft. "
                    "Treat each one as a FAIL and fix it:\n- " + "\n- ".join(checker_problems)
                    + "\n</automatic_checker_results>\n")
    prompt = (REFLECT_PROMPT.replace("{report}", REPORT).replace("{requirements}", REQUIREMENTS)
              .replace("{draft}", draft).replace("{checker_feedback}", feedback))
    result = parse_json(generate(prompt, system=REFLECT_SYSTEM, json_mode=True))
    revised = result.get("revised_summary", "").replace("\\n", "\n").strip()  # undo double-escaped line breaks
    return result.get("critique", []), revised


def show_critique(critique):
    for item in critique:
        print(f"\n[{item.get('verdict', '?')}] {item.get('requirement', '')}")
        for label in ("evidence", "fix"):
            value = str(item.get(label, "")).strip()
            if value and value.lower() != "none":
                print(textwrap.indent(wrap(f"{label}: {value}", 94), "    "))


rounds = []
draft, problems = before, None
for round_no in range(1, 4):
    critique, revised = reflect(draft, problems)
    problems = failed_checks(revised)
    rounds.append({"round": round_no, "critique": critique, "revised": revised, "failed_checks": problems})

    show(f"REFLECTION ROUND {round_no} | self-critique of the "
         f"{'BEFORE summary' if round_no == 1 else f'round {round_no - 1} revision'}")
    show_critique(critique)
    show(f"REFLECTION ROUND {round_no} | revised summary ({len(revised.split())} words)", revised)
    print("\nAutomatic checks:", "all passed" if not problems else f"{len(problems)} failed")
    for p in problems:
        print(f"  FAIL  {p}")
    if not problems:
        break
    draft = revised

after = rounds[-1]["revised"]

## Step 3: Before vs after

In [ ]:
show(f"BEFORE ({len(before.split())} words)", before)
show(f"AFTER ({len(after.split())} words, after {len(rounds)} reflection round{'s' if len(rounds) > 1 else ''})", after)

In [ ]:
import pandas as pd

before_checks, after_checks = check_summary(before), check_summary(after)
table = pd.DataFrame([
    {"requirement check": name,
     "before": "PASS" if before_checks[name][0] else "FAIL",
     "after": "PASS" if after_checks[name][0] else "FAIL"}
    for name in before_checks
])
display(table)
print(f"Checks passed: before {sum(ok for ok, _ in before_checks.values())}/{len(before_checks)}, "
      f"after {sum(ok for ok, _ in after_checks.values())}/{len(after_checks)}")
print(f"Word count: before {len(before.split())}, after {len(after.split())}")
print("Model(s) used:", ", ".join(sorted(MODELS_USED)))

In [ ]:
# What the model itself flagged in its first self-critique
first = rounds[0]["critique"]
fails = [c for c in first if str(c.get("verdict", "")).upper().startswith("FAIL")]
print(f"In round 1 the model marked {len(fails)} of {len(first)} requirements as FAIL:")
for c in fails:
    print(f"  - {c.get('requirement')}: {wrap(str(c.get('fix', '')), 90)}")

## Prompts used (for the write-up)

In [ ]:
for name, prompt in [("Draft prompt (produces the BEFORE summary)", DRAFT_PROMPT),
                     ("Requirements used for the critique (R1-R5)", REQUIREMENTS),
                     ("Self-reflection system prompt", REFLECT_SYSTEM),
                     ("Self-reflection prompt", REFLECT_PROMPT)]:
    print(f"==================== {name} ====================\n{prompt}\n")

## Notes

**Why the "after" summary is better**
- The first-draft prompt named the reader but set no criteria, so the model had to guess what "good" meant. The comparison table above shows which requirements the before summary missed.
- The reflection prompt turned "make it better" into five specific, checkable requirements. The model had to judge its own draft against each one, quote the evidence, and state a fix before rewriting.
- The automatic checks confirm the improvement independently of the model's opinion of itself, and any problem they catch is sent back for another round.

**Prompt engineering principles used**
- **Clarity:** a named reader (Maya Chen, VP of Customer Experience) and a stated purpose (make a decision).
- **Specificity:** numbered requirements with measurable limits (120 words, exactly 3 bullets, fixed first and last lines).
- **Constraints:** no jargon acronyms, numbers and dates copied exactly from the report, plain text only.
- **Goal alignment:** the required structure leads to the decision the reader has to make, with its cost and deadline.
- **Structure:** XML-style tags separate the report, requirements and draft; the critique comes back as JSON so every requirement gets a verdict.

**Limitations:** the automatic checks can confirm numbers, format and length, but not whether every claim is phrased faithfully; that part relies on the model's R1 critique and a human read. Results vary between runs.